# MP01 — Digital Image Explorer

Welcome! A digital image is a grid of numbers. We will make our own tiny image, inspect its numbers, and connect them to colors. No dataset download or prior machine learning knowledge is needed.

**Learning goals:** explain pixels, dimensions, RGB, grayscale, bit depth, NumPy arrays, and histograms; make predictions and check them with small experiments.

**How to use this lesson:** select a cell and press **Shift+Enter** to run it and move forward. Text cells explain; code cells calculate. Run from the top. If variables become confusing, choose **Restart Kernel and Run All Cells**. A variable is a name for a value; a function is a reusable named operation. `#` begins a Python comment, `print(...)` displays a value, and `assert` checks a statement (it raises an error when false).

All images below are original deterministic teaching data, not satellite observations. The optional final section can inspect your own display image. Exercises are invitations to learn, not evidence that you have completed the project.


In [ ]:
import os
from pathlib import Path
import tempfile
# A writable cache also works in restricted teaching environments.
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mp01-mpl-cache"))
import sys
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import PIL
from PIL import Image
from image_tools import (make_synthetic_image, normalize_rgb, to_grayscale,
                         histogram_u8, brighten, load_rgb_image)
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "Matplotlib:", matplotlib.__version__, "Pillow:", PIL.__version__)


## 1. Pixels and coordinates

A **pixel** is one sample at a grid location. A grayscale image can be modeled as $I(r,c)$: row $r$, column $c$. Height $H$ counts rows; width $W$ counts columns. Indices start at zero: $0\le r<H$ and $0\le c<W$.

For RGB, $I(r,c,k)$ has a third index $k$: 0 for red, 1 for green, 2 for blue. Its shape is $(H,W,3)$. One RGB pixel has three channel samples; it is still one pixel.

**Worked example:** a 2-row, 3-column RGB image has $2\times3=6$ pixels and $6\times3=18$ channel samples. Pillow reports `size` as `(width, height)`; NumPy reports `shape` as `(height, width, channels)`.

### Experiment 1 — Build a six-pixel image
Predict the shape and bottom-right color before running. A NumPy **array** is a rectangular collection of same-type numbers. Nested brackets here mean rows, pixels, then channel values.


In [ ]:
tiny = np.array([[[255, 0, 0], [0, 255, 0], [0, 0, 255]],
                 [[255, 255, 255], [0, 0, 0], [128, 128, 128]]], dtype=np.uint8)
print("shape:", tiny.shape, "pixels:", tiny.shape[0] * tiny.shape[1], "samples:", tiny.size)
print("bottom-right pixel:", tiny[1, 2].tolist())
assert tiny.shape == (2, 3, 3) and tiny.size == 18
plt.figure(figsize=(6, 3))
plt.imshow(tiny, interpolation="nearest")
plt.xticks(range(3)); plt.yticks(range(2))
plt.xlabel("column c"); plt.ylabel("row r"); plt.title("Six pixels, enlarged for display")
plt.show()


### Experiment 2 — Generate and save a synthetic image

We create a 64-row, 96-column image with red and green gradients, a constant blue background, and six known color blocks. It uses no randomness. `image_tools.py` contains short reusable helpers; inspect them when curious. `outputs/` holds local generated files and is ignored by Git.

Predict: will Pillow's size be `(64, 96)` or `(96, 64)`?


In [ ]:
rgb = make_synthetic_image()
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)
synthetic_path = output_dir / "synthetic_rgb.png"
Image.fromarray(rgb).save(synthetic_path)
with Image.open(synthetic_path) as saved:
    print("Pillow size:", saved.size, "mode:", saved.mode)
print("NumPy shape:", rgb.shape, "dtype:", rgb.dtype, "min/max:", rgb.min(), rgb.max())
assert rgb.shape == (64, 96, 3)
plt.figure(figsize=(8, 4))
plt.imshow(rgb, interpolation="nearest"); plt.title("Original synthetic RGB image")
plt.xlabel("column"); plt.ylabel("row"); plt.show()


### Experiment 3 — Zoom into pixels and crop a region

`rgb[r, c]` gives one pixel. A colon expresses a slice: `rgb[0:3, 0:4]` selects rows 0,1,2 and columns 0,1,2,3. The stop is excluded. `rgb[:, :, 0]` means every row, every column, red channel.

**Worked example:** rows 0 through 2 and columns 0 through 3 contain $3\times4=12$ pixels. They lie in our red block. Predict every triplet.


In [ ]:
crop = rgb[0:3, 0:4].copy()
print("crop shape:", crop.shape)
print(crop)
assert np.all(crop == [255, 0, 0])
plt.figure(figsize=(6, 3)); plt.imshow(crop, interpolation="nearest")
for row in range(3):
    for col in range(4):
        plt.text(col, row, "255,0,0", ha="center", va="center", color="white", fontsize=8)
plt.xticks(range(4)); plt.yticks(range(3)); plt.title("Twelve red pixels")
plt.show()


## 2. RGB channels

In additive RGB, $(0,0,0)$ is black, $(255,255,255)$ is white, and $(255,0,0)$ is full red for our 8-bit display encoding. An RGB triplet is a color encoding, not three geographic coordinates or automatically three physical satellite bands.

### Experiment 4 — Separate the channels
Predict: which channel has a horizontal gradient? Which has a vertical gradient? Displaying a channel with `cmap="gray"` shows its numerical strength, not its original hue. Fixed `vmin=0, vmax=255` avoids automatic contrast changes.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for channel, name in enumerate(["Red", "Green", "Blue"]):
    axes[channel].imshow(rgb[:, :, channel], cmap="gray", vmin=0, vmax=255)
    axes[channel].set_title(name + " intensity"); axes[channel].axis("off")
plt.tight_layout(); plt.show()
print("First three known colors:", rgb[0, 0].tolist(), rgb[0, 16].tolist(), rgb[0, 32].tolist())


### Experiment 5 — Remove blue without changing the original

`.copy()` creates independent data. Setting one channel to zero changes colors, even though image dimensions stay the same. Predict whether the blue block becomes black and the white block becomes yellow.


In [ ]:
no_blue = rgb.copy()
no_blue[:, :, 2] = 0
assert no_blue.shape == rgb.shape
assert no_blue[0, 32].tolist() == [0, 0, 0]
assert rgb[0, 32].tolist() == [0, 0, 255]  # Original unchanged.
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, image, title in zip(axes, [rgb, no_blue], ["Original", "Blue set to zero"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## 3. Bit depth and numerical storage

A **bit** has two states. An unsigned $b$-bit channel can encode $2^b$ integer levels, numbered 0 through $2^b-1$. Eight bits give 256 levels (0–255). `uint8` means unsigned 8-bit integer. RGB with 8 bits per channel uses 24 bits per pixel, with $256^3$ possible triplets; an RGB image need not contain all of them.

The raw sample storage is $HWCb/8$ bytes, excluding metadata and compression. Our 64×96 RGB array uses $64\cdot96\cdot3=18432$ bytes. A PNG file's compressed size is a different quantity. Storage dtype also differs from sensor effective bit depth: a 12-bit sensor may store values in a 16-bit container.

### Experiment 6 — Reduce gray levels
Quantization maps continuous or finely spaced values to fewer levels. For $L=2^b$ levels spanning [0,255], we use $q(x)=255\,\mathrm{round}((L-1)x/255)/(L-1)$, then round for uint8 storage. Predict banding with two bits (four levels).


In [ ]:
ramp = np.tile(np.arange(256, dtype=np.uint8), (24, 1))
fig, axes = plt.subplots(3, 1, figsize=(9, 4))
for ax, bits in zip(axes, [8, 2, 1]):
    levels = 2 ** bits
    reduced = np.rint(np.rint(ramp.astype(np.float32) * (levels - 1) / 255) * 255 / (levels - 1)).astype(np.uint8)
    ax.imshow(reduced, cmap="gray", vmin=0, vmax=255); ax.set_title(f"{bits} bit(s): {np.unique(reduced).tolist() if bits < 8 else '256 levels'}")
    ax.axis("off")
plt.tight_layout(); plt.show()
print("raw array bytes:", rgb.nbytes, "PNG file bytes:", synthetic_path.stat().st_size)
assert rgb.nbytes == 18432


### Experiment 7 — Normalize numbers without changing the picture

Normalization here means $f=x/255$, changing uint8 [0,255] into floating-point [0,1]. **Worked example:** $128/255\approx0.50196$, not exactly 0.5. Recover the original with $x=\mathrm{round}(255f)$. Convert to float before arithmetic. Matplotlib expects RGB integers in [0,255] or RGB floats in [0,1].


In [ ]:
normalized = normalize_rgb(rgb)
restored = np.rint(normalized * 255).astype(np.uint8)
print("128/255 =", 128 / 255)
print("normalized:", normalized.dtype, normalized.min(), normalized.max())
assert np.array_equal(restored, rgb)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, image, title in zip(axes, [rgb, normalized], ["uint8 [0,255]", "float32 [0,1]"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


### Experiment 8 — Brightness, clipping, and overflow

To brighten by $a$, use $y=\min(255,\max(0,x+a))$. **Worked example:** $250+20=270$, clipped to 255. Doing the addition in uint8 wraps modulo 256 instead: $270\bmod256=14$, which is dark! We demonstrate the wrong outcome deliberately and then use a wider signed type before clipping. Predict both results.


In [ ]:
sample = np.array([250], dtype=np.uint8)
wrapped = sample + np.array([20], dtype=np.uint8)
correct = brighten(np.array([[[250, 10, 0]]], dtype=np.uint8), 20)
print("uint8 wrapped:", wrapped.tolist(), "safe RGB:", correct[0, 0].tolist())
assert wrapped[0] == 14 and correct[0, 0].tolist() == [255, 30, 20]
brighter = brighten(rgb, 40)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, image, title in zip(axes, [rgb, brighter], ["Original", "Brightness +40 (clipped)"]):
    ax.imshow(image); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## 4. Grayscale

A grayscale image has one value per pixel, shape $(H,W)$. It cannot retain all three RGB values. A simple average is $(R+G+B)/3$. Our display-gray conversion uses $Y'=0.299R'+0.587G'+0.114B'$. The prime reminds us these are encoded display values: this is an approximate **luma** signal, not linear-light luminance or measured satellite reflectance.

**Worked example:** pure red gives $0.299\times255=76.245$; pure green gives 149.685; pure blue gives 29.070. Equal RGB averages would give all three 85. We keep float32 until deliberately rounding to uint8.

### Experiment 9 — Compare two grayscale recipes
Predict which primary color looks brightest with the weighted recipe. Green receives the largest weight.


In [ ]:
gray = to_grayscale(rgb)
gray8 = np.rint(gray).clip(0, 255).astype(np.uint8)
average = rgb.astype(np.float32).mean(axis=2)
print("weighted primary grays:", to_grayscale(tiny)[0].round(3).tolist())
print("gray shape/dtype:", gray.shape, gray.dtype)
assert gray.shape == (64, 96)
np.testing.assert_allclose(to_grayscale(tiny)[0], [76.245, 149.685, 29.07], atol=0.001)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, image, title in zip(axes, [average, gray], ["Equal RGB average", "Weighted display gray"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()


## 5. Histograms

A histogram counts how often values occur; it does not record where they occur. For uint8 grayscale, define $h(v)=\sum_{r=0}^{H-1}\sum_{c=0}^{W-1}\mathbf{1}[I(r,c)=v]$, $v\in\{0,\ldots,255\}$. The indicator $\mathbf{1}$ is 1 when the condition is true and 0 otherwise. Therefore $\sum_v h(v)=HW$. The normalized histogram $p(v)=h(v)/(HW)$ sums to 1.

**Worked example:** for `[[0,0],[128,255]]`, counts are $h(0)=2$, $h(128)=1$, $h(255)=1$, all others zero. Probabilities are 0.5, 0.25, 0.25.

### Experiment 10 — Count first, then plot
Predict the sum of counts for our 64×96 image: 6144. We round weighted grayscale before counting exact integer bins.


In [ ]:
example = np.array([[0, 0], [128, 255]], dtype=np.uint8)
counts_example = histogram_u8(example)
assert counts_example[[0, 128, 255]].tolist() == [2, 1, 1]
counts = histogram_u8(gray8)
probabilities = counts / gray8.size
print("toy counts:", counts_example[[0, 128, 255]].tolist())
print("total counts:", counts.sum(), "probability sum:", probabilities.sum())
assert counts.sum() == 6144
assert np.isclose(probabilities.sum(), 1)
plt.figure(figsize=(9, 3)); plt.bar(np.arange(256), counts, width=1)
plt.xlabel("Gray intensity (0–255)"); plt.ylabel("Pixel count"); plt.title("Synthetic image grayscale histogram")
plt.tight_layout(); plt.show()


### Experiment 11 — Compare RGB channel histograms

Each RGB channel has its own distribution. Predict a strong blue peak at 64: most background blue values are constant. Each channel histogram counts 6144 samples; combining all three would count 18432 and answer a different question.


In [ ]:
plt.figure(figsize=(9, 3))
for index, color in enumerate(["red", "green", "blue"]):
    channel_counts = histogram_u8(rgb[:, :, index])
    assert channel_counts.sum() == 6144
    plt.plot(np.arange(256), channel_counts, color=color, label=color)
plt.xlabel("Channel intensity"); plt.ylabel("Count per channel"); plt.legend()
plt.title("RGB histograms"); plt.tight_layout(); plt.show()
print("Blue value 64 count:", histogram_u8(rgb[:, :, 2])[64])


### Experiment 12 — Same histogram, different arrangement

Reverse the columns of grayscale with `[:, ::-1]`. Predict: the picture changes position, but counts stay the same. A histogram cannot tell whether an object is on the left or right.


In [ ]:
mirrored = gray8[:, ::-1].copy()
assert np.array_equal(histogram_u8(gray8), histogram_u8(mirrored))
assert not np.array_equal(gray8, mirrored)
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, image, title in zip(axes, [gray8, mirrored], ["Original layout", "Mirrored layout, same histogram"]):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()
print("Histograms equal:", np.array_equal(histogram_u8(gray8), histogram_u8(mirrored)))


### Experiment 13 — Save/load round trip and NumPy memory

PNG is lossless: saving and reloading this RGB array should preserve every sample. Slices often share memory; `.copy()` makes independent storage. Predict whether changing `independent` will affect `loaded`.


In [ ]:
loaded = load_rgb_image(synthetic_path)
assert np.array_equal(loaded, rgb)
view = loaded[:2, :2]
independent = loaded[:2, :2].copy()
independent[:] = 0
print("slice shares memory:", np.shares_memory(view, loaded))
print("copy shares memory:", np.shares_memory(independent, loaded))
assert loaded[0, 0].tolist() == [255, 0, 0]
Image.fromarray(gray8).save(output_dir / "synthetic_gray.png")
print("PNG RGB round trip matched; RGB and grayscale PNGs saved locally.")


## Optional — Your own satellite display image

The default path is `None`, so this section needs no file and performs no download. Set a local path to an **opaque 8-bit RGB PNG/JPEG or simple display TIFF**, or an 8-bit grayscale/palette image. The loader applies EXIF orientation and converts grayscale/palette display images to RGB. It uses the first frame only.

This is a display-image lesson. The loader rejects transparent and high-bit-depth modes rather than silently reducing them. Pillow RGB conversion does not preserve CRS, map transforms, nodata masks, physical units, or multispectral band meanings. Do not use it to preprocess scientific satellite rasters. Export an explicit visualization separately and retain your original data; geospatial processing comes in the later remote sensing stages. Large rasters may exceed memory or Pillow's safety limits: use a small display crop. Document your image source/license privately and do not commit it.


In [ ]:
USER_IMAGE_PATH = None  # Example: Path("data/my_satellite_display.png")
if USER_IMAGE_PATH is None:
    print("Optional user image skipped: no path supplied.")
else:
    user_rgb = load_rgb_image(USER_IMAGE_PATH)
    print("User RGB shape/dtype:", user_rgb.shape, user_rgb.dtype)
    plt.figure(figsize=(8, 4)); plt.imshow(user_rgb)
    plt.title("User-provided display image"); plt.axis("off"); plt.show()
    user_gray = np.rint(to_grayscale(user_rgb)).clip(0, 255).astype(np.uint8)
    plt.figure(figsize=(8, 3)); plt.plot(histogram_u8(user_gray))
    plt.xlabel("Display gray"); plt.ylabel("Pixel count"); plt.show()


## Five practice exercises

Make a new cell for each task. Keep the original `rgb` unchanged. Predict, run, then explain the result in one or two sentences.

1. **Dimensions:** for an RGB array of shape `(20, 30, 3)`, calculate the number of pixels, channel samples, and uint8 bytes before creating it. Check with `.size` and `.nbytes`. Hint: dimensions multiply.
2. **Pixel detective:** inspect `rgb[0, 48]` and `rgb[0, 64]`. Name their colors. Explain why `rgb[64, 0]` would be outside this image. Hint: the last valid row is 63.
3. **Independent editing:** make a copy, draw a 5×7 yellow rectangle in it, and show the result. Verify that the original is unchanged. Hint: yellow is `[255,255,0]`; slices exclude the stop.
4. **Gray by hand:** calculate the equal average and weighted display-gray value for `[100,150,200]`. Compare with `to_grayscale(np.array([[[100,150,200]]], dtype=np.uint8))`. Explain why the two recipes differ.
5. **Histogram detective:** construct two different 2×3 uint8 grayscale arrays with identical histograms. Confirm equal counts and unequal arrays. Explain why histogram equality cannot prove two images are identical.

**Self-check values** (try first): exercise 1 gives 600 pixels, 1800 samples, 1800 bytes; exercise 4 gives average 150 and weighted value 140.75. Other exercises admit multiple answers. Record your own reasoning; running the notebook does not complete the exercises.

## Conclusion

A pixel is a spatial sample; RGB stores three values at each location. Array shape gives height, width, and channels, while dtype describes numeric storage. Eight-bit channels encode 256 levels. Normalizing changes representation; clipping avoids overflow; grayscale deliberately discards color information. Histograms summarize intensity counts but discard spatial layout.

The deterministic checks above connect these claims to small, inspectable examples. Our synthetic image makes no scientific claim about satellites, spectral fidelity, or model performance. The optional user-image branch has not been validated on your own imagery. Before moving on, answer the five exercises and explain the worked examples aloud.

## References

- [NumPy absolute beginners guide](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Matplotlib imshow](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)
- [Pillow image modes](https://pillow.readthedocs.io/en/stable/handbook/concepts.html#modes)
- [ITU-R BT.601](https://www.itu.int/rec/R-REC-BT.601) for the luma weighting convention; this lesson applies it to display RGB without physical radiometric calibration.
